In [ ]:
class LoanLogisticRegression:

    def __init__(self, spark, df):
        self.spark = spark
        self.df = df
        self.df_final = None
        self.train_data = None
        self.test_data = None
        self.model = None
        self.predictions = None

    # ==========================================
    # FULL PROFESSIONAL PREPROCESSING
    # ==========================================
    def preprocessing(self):

        from pyspark.ml.feature import (
            StringIndexer,
            OneHotEncoder,
            VectorAssembler,
            Imputer
        )
        from pyspark.ml import Pipeline

        if "features" in self.df.columns:
            self.df = self.df.drop("features")

        numeric_cols = [
            "Age", "Income", "LoanAmount", "Credit_Score",
            "Employment_Years", "Credit_History",
            "Has_Defaulted", "Dependents"
        ]

        categorical_cols = [
            "Gender", "Education_Level",
            "Married", "Job_Type", "Property_Area"
        ]

        imputer = Imputer(
            inputCols=numeric_cols,
            outputCols=numeric_cols
        )

        indexers = [
            StringIndexer(
                inputCol=col,
                outputCol=col + "_index",
                handleInvalid="keep"
            )
            for col in categorical_cols
        ]

        encoders = [
            OneHotEncoder(
                inputCol=col + "_index",
                outputCol=col + "_encoded"
            )
            for col in categorical_cols
        ]

        encoded_cols = [col + "_encoded" for col in categorical_cols]

        assembler = VectorAssembler(
            inputCols=numeric_cols + encoded_cols,
            outputCol="features",
            handleInvalid="keep"
        )

        pipeline = Pipeline(stages=[imputer] + indexers + encoders + [assembler])
        pipeline_model = pipeline.fit(self.df)

        self.df_final = pipeline_model.transform(self.df)

        print("Preprocessing Complete Successfully")
        print("Total Rows After Processing:", self.df_final.count())

    # ==========================================
    # Split
    # ==========================================
    def split(self):
        self.train_data, self.test_data = self.df_final.randomSplit([0.8, 0.2], seed=42)
        print("Train Count:", self.train_data.count())
        print("Test Count:", self.test_data.count())

    # ==========================================
    # Logistic Regression Definition + Training
    # ==========================================
    def train(self):

        from pyspark.ml.classification import LogisticRegression

        lr = LogisticRegression(
            featuresCol="features",
            labelCol="label",
            maxIter=100,
            regParam=0.01,
            elasticNetParam=0.0
        )

        print("Logistic Regression Model Initialized")

        self.model = lr.fit(self.train_data)

        print("Training Complete")

    # ==========================================
    # Prediction
    # ==========================================
    def predict(self):

        self.predictions = self.model.transform(self.test_data)

        print("Prediction Complete")
        self.predictions.select("label", "prediction", "probability").show(10)

    # ==========================================
    # Evaluation
    # ==========================================
    def evaluate(self):

        from pyspark.ml.evaluation import (
            MulticlassClassificationEvaluator,
            BinaryClassificationEvaluator
        )

        accuracy = MulticlassClassificationEvaluator(
            labelCol="label",
            predictionCol="prediction",
            metricName="accuracy"
        ).evaluate(self.predictions)

        f1 = MulticlassClassificationEvaluator(
            labelCol="label",
            predictionCol="prediction",
            metricName="f1"
        ).evaluate(self.predictions)

        precision = MulticlassClassificationEvaluator(
            labelCol="label",
            predictionCol="prediction",
            metricName="weightedPrecision"
        ).evaluate(self.predictions)

        recall = MulticlassClassificationEvaluator(
            labelCol="label",
            predictionCol="prediction",
            metricName="weightedRecall"
        ).evaluate(self.predictions)

        auc = BinaryClassificationEvaluator(
            labelCol="label",
            rawPredictionCol="rawPrediction",
            metricName="areaUnderROC"
        ).evaluate(self.predictions)

        print("====== FINAL EVALUATION ======")
        print("Accuracy :", accuracy)
        print("F1 Score :", f1)
        print("Precision:", precision)
        print("Recall   :", recall)
        print("AUC      :", auc)

        self.metrics = [accuracy, f1, precision, recall, auc]

    # ==========================================
    # Evaluation Visualization
    # ==========================================
    def plot_metrics(self):

        import matplotlib.pyplot as plt

        metrics = ["Accuracy", "F1", "Precision", "Recall", "AUC"]

        plt.figure()
        plt.bar(metrics, self.metrics)
        plt.title("Evaluation Metrics")
        plt.xlabel("Metrics")
        plt.ylabel("Score")
        plt.show()

    # ==========================================
    # Confusion Matrix
    # ==========================================
    def confusion_matrix(self):

        from sklearn.metrics import confusion_matrix
        import numpy as np
        import matplotlib.pyplot as plt

        y_true = self.predictions.select("label").toPandas().squeeze()
        y_pred = self.predictions.select("prediction").toPandas().squeeze()

        cm = confusion_matrix(y_true, y_pred)

        plt.figure()
        plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
        plt.title("Confusion Matrix")
        plt.colorbar()

        classes = np.unique(y_true)

        plt.xticks(np.arange(len(classes)), classes)
        plt.yticks(np.arange(len(classes)), classes)

        plt.xlabel("Predicted Label")
        plt.ylabel("True Label")

        for i in range(len(classes)):
            for j in range(len(classes)):
                plt.text(j, i, cm[i, j],
                         ha="center",
                         va="center",
                         color="white" if cm[i, j] > cm.max() / 2 else "black")

        plt.show()

    # ==========================================
    # ROC + PR Curve
    # ==========================================
    def roc_pr_curve(self):

        import matplotlib.pyplot as plt

        training_summary = self.model.summary

        roc = training_summary.roc.toPandas()

        plt.figure()
        plt.plot(roc['FPR'], roc['TPR'])
        plt.plot([0,1],[0,1],'--')
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title("ROC Curve")
        plt.show()

        pr = training_summary.pr.toPandas()

        plt.figure()
        plt.plot(pr['recall'], pr['precision'])
        plt.xlabel("Recall")
        plt.ylabel("Precision")
        plt.title("Precision-Recall Curve")
        plt.show()

    # ==========================================
    # Feature Importance
    # ==========================================
    def feature_importance(self):

        import pandas as pd
        import matplotlib.pyplot as plt

        coefficients = self.model.coefficients.toArray()

        feature_names = []

        attrs = self.df_final.schema['features'].metadata['ml_attr']['attrs']

        for attr_type in ['numeric', 'binary', 'nominal']:
            if attr_type in attrs:
                for attr in attrs[attr_type]:
                    feature_names.append(attr['name'])

        feature_importance = pd.DataFrame({
            "Feature": feature_names,
            "Coefficient": coefficients
        })

        feature_importance = feature_importance.sort_values(
            by="Coefficient",
            ascending=False
        )

        print(feature_importance)

        plt.figure()
        plt.barh(feature_importance["Feature"],
                 feature_importance["Coefficient"])
        plt.title("Feature Importance")
        plt.xlabel("Coefficient Value")
        plt.show()

    # ==========================================
    # Save Model
    # ==========================================
    def save_model(self):
        self.model.save("spark_logistic_model")
        print("Model Saved Successfully")



 #model_obj = LoanLogisticRegression(spark, df)

 # model_obj.preprocessing()
 # model_obj.split()
  #model_obj.train()
  #model_obj.predict()
 # model_obj.evaluate()
  #model_obj.plot_metrics()
 # model_obj.confusion_matrix()
  #model_obj.roc_pr_curve()
 # model_obj.feature_importance()
  #model_obj.save_model()